# 第 2 讲 · 在星禾仿真里跑通最小闭环

贯通实验的第一步：读关节、发目标、走一圈闭环、记下回合。环境是星禾 SO-101 仿真，任务是把 4 cm 方块放进料盒。
代码每一行都有中文注释；输出存在 `out/` 里。

## V0 环境检查

In [ ]:
# 看显卡：仿真用显卡渲染相机画面
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import sys                                   # 系统模块
print(sys.version.split()[0])                # 打印 Python 版本号

In [ ]:
import gymnasium as gym                      # 仿真环境的标准接口
import so101_sim                             # 星禾仿真：导入时三个场景就注册好了
print([n for n in gym.registry if n.startswith("SO101PickPlace")])   # 列出星禾的三个场景

In [ ]:
import matplotlib.pyplot as plt              # 画图工具
env = gym.make("SO101PickPlaceCube40-v1",    # 建环境：把 4 cm 方块放进料盒
               obs_mode="rgb",                      # 观测：两路相机的彩色画面，加关节角
               control_mode="pd_joint_pos")         # 动作：每个关节的目标角度
obs, info = env.reset(seed=1)                # 复位；固定随机种子，每次摆放都一样
top = obs["sensor_data"]["top"]["rgb"][0].cpu().numpy()   # 顶部相机这一帧，480×640×3
plt.imsave("out/02-top-reset.png", top)      # 存成图片，课件里要用
plt.imshow(top)                              # 在这里显示出来
plt.axis("off")                              # 不画坐标轴
plt.show()                                   # 显示

## V1 读关节

In [ ]:
joint_names = [j.name for j in env.unwrapped.agent.robot.active_joints]   # 取出 6 个关节的名字
print(joint_names)                           # 名字的顺序是固定的

In [ ]:
q = obs["agent"]["noisy_qpos"][0].numpy()    # 读状态：观测里的关节角，单位是弧度
for name, angle in zip(joint_names, q):      # 逐个关节
    print(f"{name:14s} {angle:+.3f} rad")     # 打印名字和角度

## V2 两套口径

In [ ]:
import numpy as np                           # 数值计算
from so101_sim.robots.so101_base.so101 import grip_pct_from_rad, grip_rad_from_pct   # 夹爪换算

In [ ]:
def to_real(q_rad):                          # 仿真口径（弧度）换成真机口径
    real = np.degrees(q_rad)                 # 六个量先都换成度
    real[5] = grip_pct_from_rad(q_rad[5])    # 夹爪改成 0–100 的行程百分比
    return real                              # 返回真机口径的六个数

def to_sim(real):                            # 真机口径换回仿真口径：下发目标时要用
    q_rad = np.radians(real)                 # 六个量先都换成弧度
    q_rad[5] = grip_rad_from_pct(real[5])    # 夹爪从行程百分比换回弧度
    return q_rad                             # 返回仿真口径的六个数

In [ ]:
q_real = to_real(q)                          # 当前关节角换成真机口径
for name, a, b in zip(joint_names, q, q_real):   # 两套口径并排打印
    unit = "%" if name == "gripper" else "°"      # 夹爪的单位是 %，其余是度
    print(f"{name:14s} {a:+7.3f} rad  {b:+8.2f} {unit}")   # 一行一个关节
print("换回弧度，和原来最大相差", abs(to_sim(q_real) - q).max())   # 来回换算不丢数

## V3 比例闭环

In [ ]:
K, DELTA = 0.3, 0.15                         # 增益 k；单步上限 Δ，单位弧度
EPS_DEG, MAX_STEPS = 1.0, 100                # 到达阈值，单位度；最多走的步数

In [ ]:
def rounded(x):                              # 记录里的数保留四位小数
    return np.round(x.astype(float), 4).tolist()      # 先转成 Python 的浮点数，再取四位

def policy(q, q_star):                       # 比例策略：离目标越近，动作越小
    return q + np.clip(K * (q_star - q), -DELTA, DELTA)   # 当前状态加上限幅后的 k 倍误差

def task_status(q, q_star, step):            # 任务状态：只由新状态判定
    if np.degrees(np.abs(q_star - q)).max() < EPS_DEG:    # 所有关节的最大误差小于阈值
        return "reached"                     # 到达
    return "timeout" if step >= MAX_STEPS else "running"  # 步数用完是超时，否则还在运行

In [ ]:
def run_loop(q_star, steps=MAX_STEPS):       # 跑一圈最小闭环
    obs, _ = env.reset(seed=1)               # 复位：每圈从同一个起点出发
    q = obs["agent"]["noisy_qpos"][0].numpy()    # 读状态
    episode = []                             # 回合记录，一步一条
    for step in range(1, steps + 1):         # 最多走 steps 步
        action = policy(q, q_star)           # 生成动作：下一步的目标关节角
        obs, *_ = env.step(action[None])     # 执行：下发动作，仿真走 1/30 秒
        q = obs["agent"]["noisy_qpos"][0].numpy()    # 读新状态
        status = task_status(q, q_star, step)        # 由新状态判定任务状态
        episode.append({"timestamp": round(step / 30, 3),   # 记录：时刻
                        "joint_state": rounded(q),          # 关节状态
                        "target_joint": rounded(q_star),    # 目标
                        "action_command": rounded(action),  # 动作
                        "task_status": status,              # 任务状态
                        "success": status == "reached",     # 是否成功
                        "error": None})                     # 错误信息
        if status != "running":              # 到达或超时
            break                            # 结束这一圈
    return episode, obs                      # 返回记录和最后的观测

In [ ]:
q_star = q.copy()                            # 目标先取起点的关节角：其余关节不动
q_star[0] += np.radians(20)                  # 肩回转 +20°：先写成度，再换成弧度
q_star[2] += np.radians(-15)                 # 肘 −15°
episode, obs = run_loop(q_star)              # 跑一圈闭环
print(f"用了 {len(episode)} 步，最后状态：{episode[-1]['task_status']}")   # 打印步数和最后状态

In [ ]:
target = np.array([r["target_joint"] for r in episode])   # 每步的目标
state = np.array([r["joint_state"] for r in episode])     # 每步的关节状态
plt.figure(figsize=(10, 4.5))                # 画布大一点
plt.plot(np.degrees(target - state), label=joint_names)   # 每个关节一条误差曲线，单位度
plt.xlabel("step")                           # 横轴：步数
plt.ylabel("error (deg)")                    # 纵轴：误差
plt.legend()                                 # 图例
plt.savefig("out/02-error-curves.png", dpi=150)   # 存成图片
plt.show()                                   # 显示

## V4 故意写错

In [ ]:
def top_view(obs):                           # 取顶部相机的一帧
    return obs["sensor_data"]["top"]["rgb"][0].cpu().numpy()   # 480×640×3 的数组
offset_deg = np.array([20, 0, -15, 0, 0, 0])      # 目标的偏移，单位是度：肩回转 +20，肘 −15

In [ ]:
def compare(target, name):                   # 同一个目标走 30 步，前后各拍一张
    before = top_view(env.reset(seed=1)[0])  # 出发前的画面
    after = top_view(run_loop(target, steps=30)[1])   # 走 30 步（1 秒）后的画面
    plt.imsave(f"out/02-{name}-error.png", after)     # 存下走完的画面，课件里要用
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))   # 左右两张图
    for ax, image, when in zip(axes, [before, after], ["before", "after"]):   # 逐张画
        ax.imshow(image)                     # 显示画面
        ax.set_title(f"{name} error: {when}")   # 标题
        ax.axis("off")                       # 不画坐标轴
    plt.show()                               # 显示

In [ ]:
compare(q + offset_deg, "unit")              # 错误一：把度数当成弧度加上去

In [ ]:
compare(q_star[::-1], "order")               # 错误二：目标没错，但关节顺序倒了过来

## V5 记录回合

In [ ]:
import json                                  # 写 JSON 要用
episode, _ = run_loop(q_star)                # 重新走一圈正确的闭环，得到每一步的记录
record = {"robot": "SO101 sim",              # 回合信息：机器人
          "joint_names": joint_names,        # 关节名，也是数组里的顺序
          "joint_unit": "rad",               # 关节角的单位
          "control_hz": 30,                  # 控制频率
          "steps": episode}                  # 每一步一条记录
with open("out/02-episode.json", "w") as f:  # 打开文件准备写
    json.dump(record, f, indent=1)           # 写成 JSON

In [ ]:
with open("out/02-episode.json") as f:       # 把文件读回来
    saved = json.load(f)                     # 解析成字典
print(saved["joint_unit"], saved["control_hz"], len(saved["steps"]))   # 单位、频率、步数
print(saved["steps"][0])                     # 第一步
print(saved["steps"][-1])                    # 最后一步